# Dataset download and setup

**Document image unwarping | 18 November 2026 | Notebook 1 of 2**

Acquire and audit the datasets needed for the project, then write reliable inventories for the separate EDA notebook. **No model inference, training, or model-performance evaluation occurs here.**

| Dataset | Role | Source | Expected content |
|---|---|---|---|
| UVDoc Final | Supervised training corpus; validation split to be designed later | [UVDoc-Dataset](https://github.com/tanguymagne/UVDoc-Dataset) | Images, 2D/3D grids, metadata and auxiliary annotations |
| UVDoc Benchmark | Held-out annotated benchmark | Same official release | Separate annotated benchmark archive |
| DocUNet | Real-world test benchmark | Existing `dataset/DocUNet/` | 130 photographs, 130 crops, 65 flat references |
| DIR300 | Independent real-world test benchmark | [DocGeoNet](https://github.com/fh2019ustc/DocGeoNet) | 300 distorted/reference pairs |

**Read before running:** The UVDoc Final download is approximately **27 GB compressed**, as checked by the project owner; the extracted size is not yet verified. Ensure adequate Google Drive quota and sufficient temporary/extraction space. Run the cells in order and inspect each stage's output. Large downloads and Google Drive extraction can exceed a Colab session. Each dataset has its own download/extraction stage so you can resume without repeating completed work.

The notebook never deletes dataset originals, and a folder merely existing does **not** count as verification. The EDA notebook will perform deeper image and geometry analysis and decide the training/validation split.

## 1. Environment and project paths

Mount the existing Drive folder. On Windows, the same dataset is accessible under `G:\My Drive\Document Unwarping`. No nested repository is created. The notebook saves new data only in `dataset/` and its own machine-readable reports in `dataset/_metadata/`. A GPU is not required.

In [1]:
from pathlib import Path
import json
import os
import platform
import shutil
import sys

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
if (IN_COLAB) :
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/Document Unwarping") if IN_COLAB else Path(os.environ.get("DOCUMENT_UNWARPING_ROOT", r"G:\My Drive\Document Unwarping"))
if (not PROJECT_ROOT.is_dir()) :
    raise FileNotFoundError(f"Project root not found: {PROJECT_ROOT}")

DATASET_ROOT  = PROJECT_ROOT / "dataset"
METADATA_ROOT = DATASET_ROOT / "_metadata"
DOCUNET_ROOT  = DATASET_ROOT / "DocUNet"
DIR300_ROOT   = DATASET_ROOT / "DIR300"
UVDOC_ROOT    = DATASET_ROOT / "UVDoc"

DATASET_ROOT.mkdir(exist_ok=True)
METADATA_ROOT.mkdir(exist_ok=True)

print(f"Python:       {sys.version.split()[0]}")
print(f"Platform:     {platform.platform()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset root: {DATASET_ROOT}")
print("Existing datasets:", sorted(p.name for p in DATASET_ROOT.iterdir() if p.is_dir()))

usage = shutil.disk_usage(PROJECT_ROOT)
print(f"Mounted filesystem reports {usage.free / (1024 ** 3):.1f} GiB free")
print("Google Drive quota is NOT reliably represented by mounted filesystem free space; check Drive storage separately.")

Mounted at /content/drive
Python:       3.13.18
Platform:     Linux-6.6.159+-x86_64-with-glibc2.39
Project root: /content/drive/MyDrive/Document Unwarping
Dataset root: /content/drive/MyDrive/Document Unwarping/dataset
Existing datasets: ['DIR300', 'DocUNet', 'UVDoc', '_metadata']
Mounted filesystem reports 214.4 GiB free
Google Drive quota is NOT reliably represented by mounted filesystem free space; check Drive storage separately.


The download notebook uses a few small libraries. `gdown` version 6 or later supports public Drive folders containing more than 50 files. We install it in the notebook environment; this does not modify the project's tracked requirements files.

In [2]:
%pip -q install "gdown>=6.0,<7" "requests>=2.31" "tqdm>=4.66" "pandas>=2.0" "Pillow>=10" "h5py>=3.10"

In [3]:
import collections
import csv
import datetime as dt
import hashlib
import importlib.metadata
import re
import stat
import urllib.parse
import zipfile
import zlib

import gdown
import h5py
import pandas as pd
import requests
from PIL import Image
from tqdm.auto import tqdm

for package in ["gdown", "requests", "pandas", "Pillow", "h5py", "tqdm"] :
    print(f"{package:10s} {importlib.metadata.version(package)}")

print("Dataset directory:")
for entry in sorted(DATASET_ROOT.iterdir()) :
    print(f"  {entry.name}/" if entry.is_dir() else f"  {entry.name}")

gdown      6.5.2
requests   2.32.4
pandas     2.2.3
Pillow     11.3.0
h5py       3.14.0
tqdm       4.67.3
Dataset directory:
  DIR300/
  DocUNet/
  UVDoc/
  _metadata/


## 2. Official sources and acquisition settings

The ETH Zurich links below are the exact archives linked from the [UVDoc dataset authors' README](https://github.com/tanguymagne/UVDoc-Dataset). DIR300 is a **public Google Drive folder**, not a single ZIP; its folder ID comes from the [DocGeoNet authors](https://github.com/fh2019ustc/DocGeoNet#dir300-test-set).

The download flags are intentional. Set a flag to `False` if that dataset has already been acquired or if you want to run only the structural audit first. Extracted datasets are stored in `dataset/UVDoc/`; DocUNet stays in its current folder.

In [4]:
UVDOC_FINAL_URL     = "https://igl.ethz.ch/projects/uvdoc/UVDoc_final.zip"
UVDOC_BENCHMARK_URL = "https://igl.ethz.ch/projects/uvdoc/UVDoc_benchmark.zip"
DIR300_FOLDER_URL   = "https://drive.google.com/drive/folders/1yySouQQ3BlH7OjnUhq4CLuvpX2KXtifX?usp=sharing"

UVDOC_ARCHIVES = UVDOC_ROOT / "_archives"
UVDOC_FINAL    = UVDOC_ROOT / "UVDoc_final"
UVDOC_BENCH    = UVDOC_ROOT / "UVDoc_benchmark"
DIR300_SOURCE  = DIR300_ROOT / "_source"

DOWNLOAD_UVDOC_FINAL     = False
DOWNLOAD_UVDOC_BENCHMARK = False
DOWNLOAD_DIR300          = False
EXTRACT_UVDOC_FINAL      = False
EXTRACT_UVDOC_BENCHMARK  = False

UVDOC_ARCHIVES.mkdir(parents=True, exist_ok=True)

sources = {
    "uvdoc_final"     : {"url" : UVDOC_FINAL_URL, "role" : "training"},
    "uvdoc_benchmark" : {"url" : UVDOC_BENCHMARK_URL, "role" : "held_out_annotated_test"},
    "docunet"         : {"url" : "existing local dataset", "role" : "real_world_test"},
    "dir300"          : {"url" : DIR300_FOLDER_URL, "role" : "external_real_world_test"},
}
print(pd.DataFrame.from_dict(sources, orient="index").to_string())

                                                                                                  url                      role
uvdoc_final                                        https://igl.ethz.ch/projects/uvdoc/UVDoc_final.zip                  training
uvdoc_benchmark                                https://igl.ethz.ch/projects/uvdoc/UVDoc_benchmark.zip   held_out_annotated_test
docunet                                                                        existing local dataset           real_world_test
dir300           https://drive.google.com/drive/folders/1yySouQQ3BlH7OjnUhq4CLuvpX2KXtifX?usp=sharing  external_real_world_test


## 3. Large HTTP archive downloads

UVDoc downloads write to `*.part` and save the remote identity alongside the partial file. **Resume requires a server validator** (`ETag` or `Last-Modified`) and a matching HTTP `206 Content-Range`. A server that ignores range requests is rejected rather than silently appending another complete download. If an interrupted transfer cannot be resumed safely, leave the partial file in place and decide explicitly whether to restart.

A completed transfer is checked for a readable ZIP central directory. **Full CRC verification happens during extraction**, not merely on download. If the remote server provides no content length, the archive cannot be considered complete solely from the reported number of received bytes; structural ZIP inspection is still required.

In [5]:
def write_json(path : Path, payload : dict) :
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(path)


def download_zip(url : str, destination : Path) :
    destination.parent.mkdir(parents=True, exist_ok=True)
    if (destination.exists()) :
        with zipfile.ZipFile(destination) as archive :
            archive.infolist()
        print(f"Archive already present with readable ZIP directory: {destination}")
        return destination

    partial = destination.with_name(destination.name + ".part")
    sidecar = destination.with_name(destination.name + ".part.json")
    offset  = partial.stat().st_size if partial.exists() else 0
    headers = {}
    state   = {}

    if (offset) :
        if (not sidecar.exists()) :
            raise RuntimeError(f"Cannot safely resume {partial}: missing transfer metadata")
        state = json.loads(sidecar.read_text(encoding="utf-8"))
        validator = state.get("etag") or state.get("last_modified")
        if (state.get("url") != url or not validator) :
            raise RuntimeError(f"Cannot safely resume {partial}: URL or remote identity is unverified")
        headers = {"Range" : f"bytes={offset}-", "If-Range" : validator}

    with requests.get(url, headers=headers, stream=True, timeout=(30, 120), allow_redirects=True) as response :
        if (offset and response.status_code != 206) :
            raise RuntimeError(f"Server did not honor safe resume for {destination.name}: HTTP {response.status_code}; partial preserved")
        response.raise_for_status()
        if (not offset and response.status_code != 200) :
            raise RuntimeError(f"Unexpected HTTP {response.status_code} for {url}")

        content_type = response.headers.get("Content-Type", "").lower()
        if ("text/html" in content_type) :
            raise RuntimeError(f"Expected a ZIP, received HTML from {response.url}")

        if (offset) :
            match = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+|\*)", response.headers.get("Content-Range", ""))
            if (not match or int(match.group(1)) != offset or match.group(3) == "*") :
                raise RuntimeError("Invalid Content-Range: refusing to combine potentially different downloads")
            total = int(match.group(3))
            if (state.get("total") and int(state["total"]) != total) :
                raise RuntimeError("Remote ZIP size changed since the partial download")
            for key, header in [("etag", "ETag"), ("last_modified", "Last-Modified")] :
                if (state.get(key) and response.headers.get(header) and state[key] != response.headers[header]) :
                    raise RuntimeError(f"Remote ZIP {key} changed since the partial download")
            length = response.headers.get("Content-Length")
            if (length and int(length) != total - offset) :
                raise RuntimeError("Content-Length and Content-Range do not agree")
        else :
            length = response.headers.get("Content-Length")
            total  = int(length) if length else None
            state  = {
                "url"           : url,
                "etag"          : response.headers.get("ETag"),
                "last_modified" : response.headers.get("Last-Modified"),
                "total"         : total,
            }
            write_json(sidecar, state)

        mode = "ab" if offset else "wb"
        with partial.open(mode) as output, tqdm(total=total, initial=offset, unit="B", unit_scale=True, desc=destination.name) as bar :
            for block in response.iter_content(chunk_size=8 * 1024 * 1024) :
                if (block) :
                    output.write(block)
                    bar.update(len(block))

    actual = partial.stat().st_size
    if (total is not None and actual != total) :
        raise RuntimeError(f"Incomplete download: received {actual:,} of {total:,} bytes; partial preserved")
    with zipfile.ZipFile(partial) as archive :
        if (not archive.infolist()) :
            raise RuntimeError("The downloaded ZIP contains no entries")

    partial.replace(destination)
    sidecar.unlink(missing_ok=True)
    print(f"Saved structurally readable ZIP: {destination} ({actual / (1024 ** 3):.2f} GiB). CRC verification pending extraction.")
    return destination

Run each archive download in its own cell. The existing DocUNet archive is **never** fetched again. A failed transfer leaves its `.part` file in Google Drive; rerun its cell after confirming that the server supports resume.

In [6]:
if (DOWNLOAD_UVDOC_FINAL) :
    download_zip(UVDOC_FINAL_URL, UVDOC_ARCHIVES / "UVDoc_final.zip")
else :
    print("UVDoc Final download skipped by configuration")

UVDoc Final download skipped by configuration


In [7]:
if (DOWNLOAD_UVDOC_BENCHMARK) :
    download_zip(UVDOC_BENCHMARK_URL, UVDOC_ARCHIVES / "UVDoc_benchmark.zip")
else :
    print("UVDoc Benchmark download skipped by configuration")

UVDoc Benchmark download skipped by configuration


## 4. Inspect archives and extract safely

Inspect archive contents **before** extraction. The extractor rejects absolute paths, parent traversal, duplicate output names and symbolic links. Extraction occurs in a persistent staging folder, with per-file `.partial` writes and CRC checks supplied by Python's ZIP reader. On rerun it verifies the CRC of staged files before skipping them. Only when every entry has been read successfully is the staged folder renamed to the final destination.

The total uncompressed size can greatly exceed ZIP size. Check your Drive quota manually before running extraction. **No existing final directory is overwritten.**

In [8]:
def inspect_zip(path : Path, expected_name : str) :
    with zipfile.ZipFile(path) as archive :
        members = archive.infolist()
        total   = sum(member.file_size for member in members if not member.is_dir())
        files   = [member for member in members if not member.is_dir()]
        print(f"{path.name}: {len(files):,} files, {total / (1024 ** 3):.2f} GiB uncompressed")
        print("Top-level entries:", collections.Counter(name.filename.split("/")[0] for name in files).most_common(12))
        for member in sorted(files, key=lambda item : item.file_size, reverse=True)[ : 5] :
            print(f"  {member.file_size / (1024 ** 3):.2f} GiB  {member.filename}")
        if (not files) :
            raise ValueError(f"ZIP is empty: {path}")
    return total


for archive_name, folder_name in [("UVDoc_final.zip", "UVDoc_final"), ("UVDoc_benchmark.zip", "UVDoc_benchmark")] :
    archive_path = UVDOC_ARCHIVES / archive_name
    if (archive_path.exists()) :
        inspect_zip(archive_path, folder_name)
    else :
        print(f"Missing archive: {archive_path}")

UVDoc_final.zip: 182,480 files, 28.75 GiB uncompressed
Top-level entries: [('__MACOSX', 91246), ('UVDoc_final', 91234)]
  0.00 GiB  UVDoc_final/uvmap/01_00047_1_1_1.mat
  0.00 GiB  UVDoc_final/uvmap/01_00047_1_0_1.mat
  0.00 GiB  UVDoc_final/uvmap/01_00022_1_1_1.mat
  0.00 GiB  UVDoc_final/uvmap/01_00022_1_0_1.mat
  0.00 GiB  UVDoc_final/uvmap/01_00047_1_1_0.mat
UVDoc_benchmark.zip: 601 files, 0.77 GiB uncompressed
Top-level entries: [('UVDoc_benchmark', 601)]
  0.01 GiB  UVDoc_benchmark/uvmap/09_00027_1_0_0.mat
  0.01 GiB  UVDoc_benchmark/uvmap/11_00004_1_0_0.mat
  0.01 GiB  UVDoc_benchmark/uvmap/11_00060_1_0_0.mat
  0.01 GiB  UVDoc_benchmark/uvmap/09_00030_1_0_0.mat
  0.01 GiB  UVDoc_benchmark/uvmap/11_00013_1_0_0.mat


In [9]:
def archive_members(archive : zipfile.ZipFile, expected_name : str) :
    entries = archive.infolist()
    files   = [item for item in entries if not item.is_dir()]
    if (not files) :
        raise ValueError("The archive contains no files")

    known_directories = {"img", "grid2d", "grid3d", "metadata_sample", "metadata_geom", "seg", "uvmap", "wc", "textures", "img_geom", "warped_textures", "dist", "gt"}
    tops = {item.filename.split("/")[0] for item in files}
    prefix = next(iter(tops)) if len(tops) == 1 and next(iter(tops)).lower() not in known_directories else None

    output = []
    seen   = set()
    for member in entries :
        name = member.filename
        if (name.startswith("/") or "\\" in name or "\x00" in name or re.match(r"^[A-Za-z]:", name)) :
            raise ValueError(f"Unsafe ZIP member: {name}")
        parts = [part for part in name.split("/") if part]
        if (".." in parts or any(part == "." for part in parts)) :
            raise ValueError(f"Unsafe relative ZIP member: {name}")
        if ((member.external_attr >> 16) & 0o170000 == stat.S_IFLNK) :
            raise ValueError(f"ZIP contains a symbolic link: {name}")
        if (prefix and parts and parts[0] == prefix) :
            parts = parts[1 : ]
        if (not parts or member.is_dir()) :
            continue
        relative = Path(*parts)
        normalized = relative.as_posix().casefold()
        if (normalized in seen) :
            raise ValueError(f"Duplicate ZIP output path: {relative}")
        seen.add(normalized)
        output.append((member, relative))

    return output


def file_matches_member(path : Path, member : zipfile.ZipInfo) :
    if (not path.is_file() or path.stat().st_size != member.file_size) :
        return False
    checksum = 0
    with path.open("rb") as source :
        for block in iter(lambda : source.read(8 * 1024 * 1024), b"") :
            checksum = zlib.crc32(block, checksum)
    return (checksum & 0xFFFFFFFF) == member.CRC


def extract_zip_safely(archive_path : Path, destination : Path, required_folders : tuple[str, ...] = ()) :
    if (destination.exists()) :
        print(f"Final destination already exists, not overwriting: {destination}")
        return
    if (not archive_path.is_file()) :
        raise FileNotFoundError(archive_path)

    staging = destination.parent / f"_staging_{destination.name}"
    staging.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive :
        members = archive_members(archive, destination.name)
        size    = sum(info.file_size for info, _ in members)
        print(f"Extracting {len(members):,} files ({size / (1024 ** 3):.2f} GiB) to {staging}")
        print("Ensure Google Drive has enough free quota. Existing staged files are checked before reuse.")
        with tqdm(total=size, unit="B", unit_scale=True, desc=destination.name) as bar :
            for info, relative in members :
                output = staging / relative
                if (file_matches_member(output, info)) :
                    bar.update(info.file_size)
                    continue
                output.parent.mkdir(parents=True, exist_ok=True)
                temporary = output.with_name(output.name + ".partial")
                with archive.open(info) as source, temporary.open("wb") as target :
                    while True :
                        block = source.read(8 * 1024 * 1024)
                        if (not block) :
                            break
                        target.write(block)
                        bar.update(len(block))
                temporary.replace(output)

    missing = [name for name in required_folders if not (staging / name).is_dir()]
    if (missing) :
        raise RuntimeError(f"Extraction finished but required UVDoc folders are missing: {missing}; staging preserved")
    staging.rename(destination)
    print(f"Extraction and per-file CRC checks completed: {destination}")

In [10]:
if (EXTRACT_UVDOC_FINAL) :
    extract_zip_safely(UVDOC_ARCHIVES / "UVDoc_final.zip", UVDOC_FINAL, required_folders=("img", "grid2d", "grid3d", "metadata_sample"))
else :
    print("UVDoc Final extraction skipped")

UVDoc Final extraction skipped


In [11]:
from pathlib import Path
import shutil
import zipfile

archive_path = Path("/content/drive/MyDrive/Document Unwarping/dataset/UVDoc/_archives/UVDoc_final.zip")

with zipfile.ZipFile(archive_path) as archive :
    members = [
        info for info in archive.infolist()
        if info.filename.startswith("UVDoc_final/")
        and not info.is_dir()
        and not Path(info.filename).name.startswith("._")
    ]
    extracted_bytes = sum(info.file_size for info in members)

archive_bytes = archive_path.stat().st_size
free_bytes    = shutil.disk_usage("/content").free
required      = archive_bytes + extracted_bytes + 5 * 1024 ** 3

print(f"Useful dataset files : {len(members):,}")
print(f"Archive size         : {archive_bytes / 1024 ** 3:.2f} GiB")
print(f"Extraction size      : {extracted_bytes / 1024 ** 3:.2f} GiB")
print(f"Free local storage   : {free_bytes / 1024 ** 3:.2f} GiB")
print(f"Required with buffer : {required / 1024 ** 3:.2f} GiB")
print(f"Enough local space   : {free_bytes >= required}")

Useful dataset files : 91,234
Archive size         : 27.51 GiB
Extraction size      : 28.73 GiB
Free local storage   : 225.70 GiB
Required with buffer : 61.24 GiB
Enough local space   : True


In [12]:
from pathlib import PurePosixPath

local_archive = Path("/content/UVDoc_final.zip")
local_root    = Path("/content/uvdoc_runtime")
local_dataset = local_root / "UVDoc_final"

if (free_bytes < required and not local_archive.is_file()) :
    raise RuntimeError("Insufficient Colab local storage for archive and extraction")

if (not local_archive.is_file() or local_archive.stat().st_size != archive_bytes) :
    partial = local_archive.with_name(local_archive.name + ".part")
    with archive_path.open("rb") as source, partial.open("wb") as target :
        with tqdm(total=archive_bytes, unit="B", unit_scale=True, desc="Copy archive") as bar :
            for block in iter(lambda : source.read(16 * 1024 ** 2), b"") :
                target.write(block)
                bar.update(len(block))
    partial.replace(local_archive)

with zipfile.ZipFile(local_archive) as archive :
    with tqdm(total=extracted_bytes, unit="B", unit_scale=True, desc="Extract UVDoc") as bar :
        for info in members :
            parts = PurePosixPath(info.filename).parts
            if (parts[0] != "UVDoc_final" or ".." in parts or "\\" in info.filename) :
                raise ValueError(f"Unsafe archive path: {info.filename}")

            output = local_root.joinpath(*parts)
            if (file_matches_member(output, info)) :
                bar.update(info.file_size)
                continue

            output.parent.mkdir(parents=True, exist_ok=True)
            temporary = output.with_name(output.name + ".partial")

            with archive.open(info) as source, temporary.open("wb") as target :
                while True :
                    block = source.read(8 * 1024 ** 2)
                    if (not block) :
                        break
                    target.write(block)
                    bar.update(len(block))

            temporary.replace(output)

required_dirs = ["img", "grid2d", "grid3d", "metadata_sample"]
missing = [name for name in required_dirs if not (local_dataset / name).is_dir()]
if (missing) :
    raise RuntimeError(f"Required dataset folders missing: {missing}")

print(f"Extracted dataset: {local_dataset}")

Copy archive:   0%|          | 0.00/29.5G [00:00<?, ?B/s]

Extract UVDoc:   0%|          | 0.00/30.9G [00:00<?, ?B/s]

Extracted dataset: /content/uvdoc_runtime/UVDoc_final


In [13]:
if (EXTRACT_UVDOC_BENCHMARK) :
    extract_zip_safely(UVDOC_ARCHIVES / "UVDoc_benchmark.zip", UVDOC_BENCH, required_folders=("img", "grid2d", "grid3d", "metadata_sample"))
else :
    print("UVDoc Benchmark extraction skipped")

UVDoc Benchmark extraction skipped


## 5. Acquire DIR300 from the official Google Drive folder

The [DocGeoNet DIR300 folder](https://drive.google.com/drive/folders/1yySouQQ3BlH7OjnUhq4CLuvpX2KXtifX?usp=sharing) is hosted as a folder, not a ZIP. Use `gdown` to list its contents **without downloading**, then download into `dataset/DIR300/_source/`. `gdown` 6+ supports folders larger than 50 files and resumable file transfers, but Google Drive sharing restrictions or quotas can still interrupt transfers. The subsequent audit, not `gdown`'s success message, determines completeness.

In [14]:
if (DOWNLOAD_DIR300) :
    listing = gdown.download_folder(url=DIR300_FOLDER_URL, output=str(DIR300_SOURCE) + os.sep, quiet=True, skip_download=True, use_cookies=False)
    print(f"Official folder listing: {len(listing)} downloadable files")
    for item in listing[ : 15] :
        print(" ", item.path)
else :
    print("DIR300 discovery skipped")

DIR300 discovery skipped


In [15]:
if (DOWNLOAD_DIR300) :
    DIR300_SOURCE.mkdir(parents=True, exist_ok=True)
    downloaded = gdown.download_folder(url=DIR300_FOLDER_URL, output=str(DIR300_SOURCE) + os.sep, quiet=False, resume=True, retries=3, use_cookies=False)
    print(f"gdown reported {len(downloaded)} files; validating pairs in the following sections")
else :
    print("DIR300 download skipped")

DIR300 download skipped


### DIR300 archives, if supplied by Google Drive

If the published Drive folder contains `.zip` files rather than already-extracted `dist/` and `gt/` directories, expand those archives independently. They are extracted to separate `_extracted_*` directories inside `dataset/DIR300/`; nothing overwrites the downloads. The following audit searches recursively for the expected image folders. If the release uses another naming convention, the audit will report that explicitly rather than guessing a mapping.

In [16]:
dir300_archives = sorted(path for path in DIR300_SOURCE.rglob("*.zip") if path.is_file()) if DIR300_SOURCE.is_dir() else []
print(f"DIR300 ZIP archives found: {len(dir300_archives)}")
for archive_path in dir300_archives :
    extract_zip_safely(archive_path, DIR300_ROOT / f"_extracted_{archive_path.stem}")
if (not dir300_archives) :
    print("No DIR300 ZIP archives require extraction")

DIR300 ZIP archives found: 0
No DIR300 ZIP archives require extraction


## 6. Inventory and validate DocUNet

The existing DocUNet filenames use three different conventions. A crop such as `10_1 copy.png` and an original such as `10_1.jpg` both correspond to reference `10.png`. This stage creates **one row per photographed view**, audits completeness, and checks that the input/reference images can be decoded. Do not rename the originals or perform any orientation correction here.

In [17]:
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


def image_files(folder : Path) :
    return sorted((path for path in folder.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES), key=lambda p : p.name.lower()) if folder.is_dir() else []


def relative_path(path : Path | None) :
    if (path is None) :
        return ""

    local_uvdoc = Path("/content/uvdoc_runtime/UVDoc_final")

    if (path.is_relative_to(PROJECT_ROOT)) :
        return path.relative_to(PROJECT_ROOT).as_posix()

    if (path.is_relative_to(local_uvdoc)) :
        return (Path("dataset/UVDoc/UVDoc_final") / path.relative_to(local_uvdoc)).as_posix()

    raise ValueError(f"Path is outside the known dataset roots: {path}")


def image_read_error(path : Path) :
    try :
        with Image.open(path) as image :
            image.verify()
        return ""
    except (OSError, ValueError, SyntaxError) as error :
        return f"{type(error).__name__}: {error}"


def audit_docunet(root : Path) :
    originals = image_files(root / "original")
    crops     = image_files(root / "crop")
    scans     = image_files(root / "scan")
    pattern   = re.compile(r"^(\d+)_([12])(?: copy)?$", flags=re.IGNORECASE)
    original_map, crop_map, scan_map = {}, {}, {}
    issues = []

    for kind, images, mapping in [("original", originals, original_map), ("crop", crops, crop_map)] :
        for path in images :
            match = pattern.fullmatch(path.stem)
            if (not match) :
                issues.append(f"Unrecognized {kind}: {path.name}")
                continue
            key = f"{int(match.group(1))}_{match.group(2)}"
            if (key in mapping) :
                issues.append(f"Duplicate {kind} ID: {key}")
            else :
                mapping[key] = path

    for path in scans :
        if (not path.stem.isdigit()) :
            issues.append(f"Unrecognized reference: {path.name}")
            continue
        key = str(int(path.stem))
        if (key in scan_map) :
            issues.append(f"Duplicate reference ID: {key}")
        else :
            scan_map[key] = path

    keys = sorted(set(original_map) | set(crop_map), key=lambda key : tuple(map(int, key.split("_"))))
    records = []
    for key in keys :
        document_id, view_id = key.split("_")
        original = original_map.get(key)
        crop     = crop_map.get(key)
        scan     = scan_map.get(document_id)
        errors   = []
        for label, path in [("original", original), ("crop", crop), ("reference", scan)] :
            if (path is None) :
                errors.append(f"missing_{label}")
            else :
                bad = image_read_error(path)
                if (bad) :
                    errors.append(f"invalid_{label}: {bad}")
        records.append({
            "dataset"        : "DocUNet",
            "sample_id"      : key,
            "document_id"    : document_id,
            "view_id"        : view_id,
            "original_path"  : relative_path(original),
            "input_path"     : relative_path(crop),
            "reference_path" : relative_path(scan),
            "input_variant"  : "crop",
            "status"         : "valid" if not errors else "; ".join(errors),
        })

    if (not (root / "eval" / "evalUnwarp.m").is_file()) :
        issues.append("Original DocUNet eval/evalUnwarp.m is missing")
    count_ok = len(originals) == 130 and len(crops) == 130 and len(scans) == 65 and len(records) == 130
    complete = count_ok and not issues and all(row["status"] == "valid" for row in records)
    report   = {"originals" : len(originals), "crops" : len(crops), "references" : len(scans), "samples" : len(records), "issues" : issues, "ready" : complete}
    return pd.DataFrame(records, columns=["dataset", "sample_id", "document_id", "view_id", "original_path", "input_path", "reference_path", "input_variant", "status"]), report


docunet_manifest, docunet_report = audit_docunet(DOCUNET_ROOT)
print(json.dumps(docunet_report, indent=2))
print(docunet_manifest.head(6).to_string(index=False))
if (not docunet_report["ready"]) :
    print("DocUNet is NOT marked ready; inspect the reported missing/invalid files before proceeding")

{
  "originals": 130,
  "crops": 130,
  "references": 65,
  "samples": 130,
  "issues": [],
  "ready": true
}
dataset sample_id document_id view_id                    original_path                        input_path             reference_path input_variant status
DocUNet       1_1           1       1 dataset/DocUNet/original/1_1.jpg dataset/DocUNet/crop/1_1 copy.png dataset/DocUNet/scan/1.png          crop  valid
DocUNet       1_2           1       2 dataset/DocUNet/original/1_2.jpg dataset/DocUNet/crop/1_2 copy.png dataset/DocUNet/scan/1.png          crop  valid
DocUNet       2_1           2       1 dataset/DocUNet/original/2_1.jpg dataset/DocUNet/crop/2_1 copy.png dataset/DocUNet/scan/2.png          crop  valid
DocUNet       2_2           2       2 dataset/DocUNet/original/2_2.jpg dataset/DocUNet/crop/2_2 copy.png dataset/DocUNet/scan/2.png          crop  valid
DocUNet       3_1           3       1 dataset/DocUNet/original/3_1.jpg dataset/DocUNet/crop/3_1 copy.png dataset/DocUNet/scan

## 7. Inventory and validate DIR300

The Drive release may contain an extra directory level. Locate its `dist/` and `gt/` directories recursively and require a **single unambiguous pair**. File stems must be numeric, and the expected IDs are `1` through `300`. This validation makes partial Google Drive downloads visible rather than silently treating them as a complete benchmark.

In [18]:
def audit_dir300(root : Path) :
    columns = ["dataset", "sample_id", "document_id", "view_id", "input_path", "reference_path", "input_variant", "status"]
    if (not root.is_dir()) :
        return pd.DataFrame(columns=columns), {"inputs" : 0, "references" : 0, "samples" : 0, "issues" : ["DIR300 directory not found"], "ready" : False}

    candidates = [directory for directory in root.rglob("*") if directory.is_dir() and directory.name.lower() in {"dist", "gt"}]
    dist_dirs  = [path for path in candidates if path.name.lower() == "dist"]
    gt_dirs    = [path for path in candidates if path.name.lower() == "gt"]
    issues     = []
    if (len(dist_dirs) != 1 or len(gt_dirs) != 1) :
        issues.append(f"Expected exactly one dist/ and gt/ folder; found dist={len(dist_dirs)}, gt={len(gt_dirs)}")
        return pd.DataFrame(columns=columns), {"inputs" : 0, "references" : 0, "samples" : 0, "issues" : issues, "ready" : False, "directories" : [str(p) for p in candidates]}

    def collect(folder : Path, kind : str) :
        result = {}
        for path in image_files(folder) :
            if (not path.stem.isdigit()) :
                issues.append(f"Non-numeric {kind} filename: {path.name}")
                continue
            number = int(path.stem)
            if (number in result) :
                issues.append(f"Duplicate {kind} ID: {number}")
            else :
                result[number] = path
        return result

    distorted = collect(dist_dirs[0], "distorted")
    reference = collect(gt_dirs[0], "reference")
    expected  = set(range(1, 301))
    if (set(distorted) != expected) :
        issues.append(f"Distorted IDs missing: {sorted(expected - set(distorted))[ : 25]} (total {len(expected - set(distorted))}); extras: {sorted(set(distorted) - expected)}")
    if (set(reference) != expected) :
        issues.append(f"Reference IDs missing: {sorted(expected - set(reference))[ : 25]} (total {len(expected - set(reference))}); extras: {sorted(set(reference) - expected)}")

    records = []
    for number in sorted(set(distorted) | set(reference)) :
        inp, ref = distorted.get(number), reference.get(number)
        errors = []
        for label, path in [("input", inp), ("reference", ref)] :
            if (path is None) :
                errors.append(f"missing_{label}")
            else :
                bad = image_read_error(path)
                if (bad) :
                    errors.append(f"invalid_{label}: {bad}")
        records.append({
            "dataset"        : "DIR300",
            "sample_id"      : str(number),
            "document_id"    : str(number),
            "view_id"        : "",
            "input_path"     : relative_path(inp),
            "reference_path" : relative_path(ref),
            "input_variant"  : "dist",
            "status"         : "valid" if not errors else "; ".join(errors),
        })

    ready  = len(distorted) == 300 and len(reference) == 300 and len(records) == 300 and not issues and all(r["status"] == "valid" for r in records)
    report = {"inputs" : len(distorted), "references" : len(reference), "samples" : len(records), "issues" : issues, "ready" : ready, "dist_dir" : relative_path(dist_dirs[0]), "gt_dir" : relative_path(gt_dirs[0])}
    return pd.DataFrame(records, columns=columns), report


dir300_manifest, dir300_report = audit_dir300(DIR300_ROOT)
print(json.dumps(dir300_report, indent=2))
print(dir300_manifest.head(6).to_string(index=False))

{
  "inputs": 300,
  "references": 300,
  "samples": 300,
  "issues": [],
  "ready": true,
  "dist_dir": "dataset/DIR300/_source/DIR300/dist",
  "gt_dir": "dataset/DIR300/_source/DIR300/gt"
}
dataset sample_id document_id view_id                               input_path                         reference_path input_variant status
 DIR300         1           1         dataset/DIR300/_source/DIR300/dist/1.png dataset/DIR300/_source/DIR300/gt/1.png          dist  valid
 DIR300         2           2         dataset/DIR300/_source/DIR300/dist/2.png dataset/DIR300/_source/DIR300/gt/2.png          dist  valid
 DIR300         3           3         dataset/DIR300/_source/DIR300/dist/3.png dataset/DIR300/_source/DIR300/gt/3.png          dist  valid
 DIR300         4           4         dataset/DIR300/_source/DIR300/dist/4.png dataset/DIR300/_source/DIR300/gt/4.png          dist  valid
 DIR300         5           5         dataset/DIR300/_source/DIR300/dist/5.png dataset/DIR300/_source/DIR300/gt/5

## 8. Inventory UVDoc Final and UVDoc Benchmark

Both releases use rendered sample IDs in `img/` and geometry IDs in the annotation files. For each image, read `metadata_sample/<sample_id>.json`, extract its actual `geom_name`, and check that the matching `grid2d/<geom_name>.mat` and `grid3d/<geom_name>.mat` exist. Check the other expected release directories and inspect a few `.mat` files and RGB images. **Do not assume `split.json` contains a train/validation split**; its meaning belongs in Notebook 2.

Full per-file image decoding and deep grid statistics belong in EDA. Here, sample image/grid reads provide an initial format check, while the metadata relationships are checked for every rendered image.

In [19]:
UVDOC_DIRS = ["img", "grid2d", "grid3d", "img_geom", "metadata_geom", "metadata_sample", "seg", "textures", "uvmap", "warped_textures", "wc"]


def audit_uvdoc(root : Path, dataset_name : str) :
    columns = ["dataset", "sample_id", "geom_name", "image_path", "grid2d_path", "grid3d_path", "metadata_path", "texture_name", "background_name", "annotation_status"]
    if (not root.is_dir()) :
        return pd.DataFrame(columns=columns), {"images" : 0, "geometries" : 0, "issues" : ["Dataset directory not found"], "ready" : False}

    directory_files = {name : [path for path in (root / name).iterdir() if path.is_file()] if (root / name).is_dir() else [] for name in UVDOC_DIRS}
    counts = {name : len(paths) for name, paths in directory_files.items()}
    extensions = {name : dict(collections.Counter(path.suffix.lower() or "<none>" for path in paths)) for name, paths in directory_files.items()}
    examples = {name : sorted(path.name for path in paths)[ : 3] for name, paths in directory_files.items()}
    missing_dirs = [name for name in UVDOC_DIRS if not (root / name).is_dir()]
    empty_dirs = [name for name, count in counts.items() if count == 0 and name not in missing_dirs]
    images = image_files(root / "img")
    issues = []
    ids    = set()
    rows   = []

    for image in tqdm(images, desc=f"{dataset_name} metadata") :
        sample_id = image.stem
        if (sample_id in ids) :
            issues.append(f"Duplicate image stem: {sample_id}")
        ids.add(sample_id)
        metadata = root / "metadata_sample" / f"{sample_id}.json"
        record = {}
        problems = []
        if (not metadata.is_file()) :
            problems.append("missing_metadata")
        else :
            try :
                record = json.loads(metadata.read_text(encoding="utf-8"))
            except (OSError, ValueError, UnicodeError) as error :
                problems.append(f"invalid_metadata: {error}")
            if (not isinstance(record, dict)) :
                problems.append("metadata_not_object")
                record = {}
        geom = record.get("geom_name", "")
        if (not isinstance(geom, str) or not geom) :
            problems.append("missing_geom_name")
            geom = ""
        grid2d = root / "grid2d" / f"{geom}.mat" if geom else None
        grid3d = root / "grid3d" / f"{geom}.mat" if geom else None
        for label, path in [("grid2d", grid2d), ("grid3d", grid3d)] :
            if (path is None or not path.is_file()) :
                problems.append(f"missing_{label}")
        rows.append({
            "dataset"           : dataset_name,
            "sample_id"         : sample_id,
            "geom_name"         : geom,
            "image_path"        : relative_path(image),
            "grid2d_path"       : relative_path(grid2d) if grid2d is not None and grid2d.is_file() else "",
            "grid3d_path"       : relative_path(grid3d) if grid3d is not None and grid3d.is_file() else "",
            "metadata_path"     : relative_path(metadata) if metadata.is_file() else "",
            "texture_name"      : str(record.get("texture_name", "")),
            "background_name"   : str(record.get("background_name", "")),
            "annotation_status" : "valid" if not problems else "; ".join(problems),
        })

    sample_errors = []
    for image in images[ : 20] :
        bad = image_read_error(image)
        if (bad) :
            sample_errors.append(f"{image.name}: {bad}")

    sample_grid = next((row for row in rows if row["annotation_status"] == "valid"), None)
    grid_info   = {}
    if (sample_grid) :
        for name in ["grid2d_path", "grid3d_path"] :
            folder = "grid2d" if name == "grid2d_path" else "grid3d"
            path   = root / folder / f"{sample_grid['geom_name']}.mat"
            try :
                with h5py.File(path, "r") as file :
                    grid_info[name] = {key : {"shape" : list(file[key].shape), "dtype" : str(file[key].dtype)} for key in file if isinstance(file[key], h5py.Dataset)}
            except (OSError, ValueError) as error :
                sample_errors.append(f"Invalid example {name}: {error}")

    image_ids = {path.stem for path in images}
    metadata_ids = {path.stem for path in (root / "metadata_sample").glob("*.json")} if (root / "metadata_sample").is_dir() else set()
    if (metadata_ids - image_ids) :
        issues.append(f"Metadata with no image: {len(metadata_ids - image_ids)} (examples {sorted(metadata_ids - image_ids)[ : 10]})")
    split_info = {}
    if ((root / "split.json").is_file()) :
        try :
            split_data = json.loads((root / "split.json").read_text(encoding="utf-8"))
            split_info = {"top_level_type" : type(split_data).__name__, "top_level_keys" : list(split_data)[ : 20] if isinstance(split_data, dict) else [], "top_level_length" : len(split_data) if hasattr(split_data, "__len__") else None}
        except (OSError, ValueError, UnicodeError) as error :
            issues.append(f"Unreadable split.json: {error}")
    else :
        issues.append("split.json missing")
    if (missing_dirs) :
        issues.append(f"Missing expected directories: {missing_dirs}")
    if (empty_dirs) :
        issues.append(f"Empty expected directories: {empty_dirs}")
    if (dataset_name == "UVDoc Benchmark" and len(images) != 50) :
        issues.append(f"Expected 50 UVDoc benchmark images, found {len(images)}")
    if (not images) :
        issues.append("No rendered images")
    invalid = sum(row["annotation_status"] != "valid" for row in rows)
    if (invalid) :
        issues.append(f"Samples with missing metadata or grids: {invalid}")
    if (sample_errors) :
        issues.extend(sample_errors)

    report = {"images" : len(images), "unique_geometries" : len(set(row["geom_name"] for row in rows if row["geom_name"])), "directory_counts" : counts, "directory_extensions" : extensions, "example_filenames" : examples, "split_file_structure" : split_info, "missing_directories" : missing_dirs, "sampled_images_checked" : min(len(images), 20), "sample_grid_structure" : grid_info, "issues" : issues, "ready" : bool(images) and not issues}
    return pd.DataFrame(rows, columns=columns), report


uvdoc_final_root = Path("/content/uvdoc_runtime/UVDoc_final")

if (not (uvdoc_final_root / "img").is_dir()) :
    uvdoc_final_root = UVDOC_FINAL

print(f"UVDoc Final data root: {uvdoc_final_root}")

uvdoc_final_manifest, uvdoc_final_report = audit_uvdoc(uvdoc_final_root, "UVDoc Final")
print(json.dumps(uvdoc_final_report, indent=2)[ : 6000])
print(uvdoc_final_manifest.head(4).to_string(index=False))

UVDoc Final data root: /content/uvdoc_runtime/UVDoc_final


UVDoc Final metadata:   0%|          | 0/20000 [00:00<?, ?it/s]

{
  "images": 20000,
  "unique_geometries": 4032,
  "directory_counts": {
    "img": 20000,
    "grid2d": 4032,
    "grid3d": 4032,
    "img_geom": 4032,
    "metadata_geom": 4032,
    "metadata_sample": 20000,
    "seg": 4032,
    "textures": 3009,
    "uvmap": 4032,
    "warped_textures": 20000,
    "wc": 4032
  },
  "directory_extensions": {
    "img": {
      ".png": 20000
    },
    "grid2d": {
      ".mat": 4032
    },
    "grid3d": {
      ".mat": 4032
    },
    "img_geom": {
      ".png": 4032
    },
    "metadata_geom": {
      ".json": 4032
    },
    "metadata_sample": {
      ".json": 20000
    },
    "seg": {
      ".mat": 4032
    },
    "textures": {
      ".png": 3009
    },
    "uvmap": {
      ".mat": 4032
    },
    "warped_textures": {
      ".png": 20000
    },
    "wc": {
      ".exr": 4032
    }
  },
  "example_filenames": {
    "img": [
      "00000.png",
      "00001.png",
      "00002.png"
    ],
    "grid2d": [
      "00_00000_1_0_0.mat",
      "00_00000_1_0

In [20]:
uvdoc_benchmark_manifest, uvdoc_benchmark_report = audit_uvdoc(UVDOC_BENCH, "UVDoc Benchmark")
print(json.dumps(uvdoc_benchmark_report, indent=2)[:6000])
print(uvdoc_benchmark_manifest.head(4).to_string(index=False))

UVDoc Benchmark metadata:   0%|          | 0/50 [00:00<?, ?it/s]

{
  "images": 50,
  "unique_geometries": 50,
  "directory_counts": {
    "img": 50,
    "grid2d": 50,
    "grid3d": 50,
    "img_geom": 50,
    "metadata_geom": 50,
    "metadata_sample": 50,
    "seg": 50,
    "textures": 50,
    "uvmap": 50,
    "warped_textures": 50,
    "wc": 50
  },
  "directory_extensions": {
    "img": {
      ".png": 50
    },
    "grid2d": {
      ".mat": 50
    },
    "grid3d": {
      ".mat": 50
    },
    "img_geom": {
      ".png": 50
    },
    "metadata_geom": {
      ".json": 50
    },
    "metadata_sample": {
      ".json": 50
    },
    "seg": {
      ".mat": 50
    },
    "textures": {
      ".png": 50
    },
    "uvmap": {
      ".mat": 50
    },
    "warped_textures": {
      ".png": 50
    },
    "wc": {
      ".exr": 50
    }
  },
  "example_filenames": {
    "img": [
      "00000.png",
      "00001.png",
      "00002.png"
    ],
    "grid2d": [
      "09_00000_1_0_0.mat",
      "09_00004_1_0_0.mat",
      "09_00005_1_0_0.mat"
    ],
    "grid3d"

## 9. Export manifests and integrity reports

Write portable **project-relative paths** and preserve the status of missing/incomplete datasets. A failed audit produces an explicit `ready = false`, not an empty dataset falsely labeled complete. These outputs are consumed by Notebook 2. The existing `documentation/docunet_manifest.json` is left untouched.

In [21]:
manifests = {
    "docunet_manifest.csv"         : docunet_manifest,
    "dir300_manifest.csv"          : dir300_manifest,
    "uvdoc_final_manifest.csv"     : uvdoc_final_manifest,
    "uvdoc_benchmark_manifest.csv" : uvdoc_benchmark_manifest,
}
for filename, frame in manifests.items() :
    output = METADATA_ROOT / filename
    frame.to_csv(output, index=False)
    print(f"Saved {output.relative_to(PROJECT_ROOT)} ({len(frame):,} rows)")

integrity = {
    "created_at_utc" : dt.datetime.now(dt.timezone.utc).isoformat(),
    "docunet"        : docunet_report,
    "dir300"         : dir300_report,
    "uvdoc_final"    : uvdoc_final_report,
    "uvdoc_benchmark": uvdoc_benchmark_report,
}
write_json(METADATA_ROOT / "integrity_report.json", integrity)

acquisition = {}
for name, info in sources.items() :
    archive = {"uvdoc_final" : UVDOC_ARCHIVES / "UVDoc_final.zip", "uvdoc_benchmark" : UVDOC_ARCHIVES / "UVDoc_benchmark.zip"}.get(name)
    folder  = {"uvdoc_final" : uvdoc_final_root, "uvdoc_benchmark" : UVDOC_BENCH, "dir300" : DIR300_ROOT, "docunet" : DOCUNET_ROOT}[name]
    acquisition[name] = {
        "source_url"     : info["url"],
        "role"           : info["role"],
        "archive_present": archive.is_file() if archive else None,
        "archive_bytes"  : archive.stat().st_size if archive and archive.is_file() else None,
        "folder_present" : folder.is_dir(),
        "structure_ready" : integrity[name]["ready"],
        "active_data_root" : str(folder),
    }
write_json(METADATA_ROOT / "acquisition.json", acquisition)

uncompressed_sizes = {}
for archive_name in ["UVDoc_final.zip", "UVDoc_benchmark.zip"] :
    path = UVDOC_ARCHIVES / archive_name
    if (path.is_file()) :
        with zipfile.ZipFile(path) as archive :
            uncompressed_sizes[archive_name] = sum(member.file_size for member in archive.infolist() if not member.is_dir())

storage = {"created_at_utc" : dt.datetime.now(dt.timezone.utc).isoformat(), "archive_sizes_bytes" : {name : values["archive_bytes"] for name, values in acquisition.items()}, "uncompressed_sizes_bytes" : uncompressed_sizes, "drive_quota_verified" : False, "notes" : "Mounted filesystem free space is not authoritative for Drive quota; compare archive and uncompressed sizes manually."}
write_json(METADATA_ROOT / "storage_report.json", storage)

print("Saved integrity_report.json, acquisition.json, and storage_report.json")

Saved dataset/_metadata/docunet_manifest.csv (130 rows)
Saved dataset/_metadata/dir300_manifest.csv (300 rows)
Saved dataset/_metadata/uvdoc_final_manifest.csv (20,000 rows)
Saved dataset/_metadata/uvdoc_benchmark_manifest.csv (50 rows)
Saved integrity_report.json, acquisition.json, and storage_report.json


## 10. Dataset readiness and handoff to EDA

Readiness here means that expected files exist, benchmark image pairs are decodable, UVDoc sample metadata resolves to 2D/3D grids, and a few example UVDoc files can be decoded. **It does not mean every UVDoc image or grid has undergone full numerical inspection.** Those deeper tests, including duplicate-content checks, geometry distributions, and train/validation split design, belong in Notebook 2. Record any blockers before proceeding.

In [22]:
readiness = pd.DataFrame([
    {"dataset" : "UVDoc Final",     "role" : "training and future validation", "samples" : len(uvdoc_final_manifest),     "ready_for_eda" : uvdoc_final_report["ready"]},
    {"dataset" : "UVDoc Benchmark", "role" : "held-out annotated test",    "samples" : len(uvdoc_benchmark_manifest), "ready_for_eda" : uvdoc_benchmark_report["ready"]},
    {"dataset" : "DocUNet",         "role" : "real-world test",            "samples" : len(docunet_manifest),         "ready_for_eda" : docunet_report["ready"]},
    {"dataset" : "DIR300",          "role" : "external real-world test",   "samples" : len(dir300_manifest),          "ready_for_eda" : dir300_report["ready"]},
])
display(readiness)

pending = readiness.loc[~readiness["ready_for_eda"], "dataset"].tolist()
if (pending) :
    print("Not yet ready:", ", ".join(pending))
    print(f"See {METADATA_ROOT / 'integrity_report.json'} for the exact failures and missing files.")
else :
    print("All four dataset inventories are structurally ready for Notebook 2.")

print("Next: comprehensive Dataset_EDA notebook, including geometry analysis and a leakage-controlled validation split.")

,dataset,role,samples,ready_for_eda
0,UVDoc Final,training and future validation,20000,True
1,UVDoc Benchmark,held-out annotated test,50,True
2,DocUNet,real-world test,130,True
3,DIR300,external real-world test,300,True


All four dataset inventories are structurally ready for Notebook 2.
Next: comprehensive Dataset_EDA notebook, including geometry analysis and a leakage-controlled validation split.
